## Treinamento de modelo customizado

Este notebook é um exemplo de referência para treinar um detector YOLO em um dataset customizado. A ideia não é substituir o seu dataset, e sim mostrar o fluxo completo para que o aluno consiga adaptar esse código ao seu problema.

Antes de treinar, é importante lembrar a diferença entre:

- classificação: responde "o que é a imagem?";
- detecção: responde "o que é, quantos são e onde estão?".

Neste exemplo, usaremos um dataset de referência no formato YOLO e mostraremos como preparar a configuração, treinar, avaliar e testar o modelo.


In [1]:
from ultralytics import YOLO

# Modelo pequeno pré-treinado: escolha apropriada para o primeiro experimento.
model = YOLO("yolo11n.pt")

## 1. Escolha o dataset

Use o dataset que você criou no Lab 13. Ele deve estar no formato YOLO e ter uma divisão entre treino e validação.

```text
meu-dataset/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

Cada imagem precisa ter um arquivo `.txt` com o mesmo nome na pasta de labels. Neste notebook, altere apenas as variáveis da próxima célula para apontar para o seu dataset e suas classes.

In [ ]:
from pathlib import Path

# Altere este caminho para a pasta raiz do dataset criado por você.
DATASET_DIR = Path("/caminho/para/meu-dataset")

# Mantenha a mesma ordem de classes usada nos arquivos .txt de anotação.
CLASS_NAMES = ["classe_0", "classe_1"]

assert (DATASET_DIR / "images" / "train").is_dir(), "Pasta images/train não encontrada. Ajuste DATASET_DIR."
assert (DATASET_DIR / "images" / "val").is_dir(), "Pasta images/val não encontrada. Ajuste DATASET_DIR."
assert (DATASET_DIR / "labels" / "train").is_dir(), "Pasta labels/train não encontrada. Ajuste DATASET_DIR."
assert (DATASET_DIR / "labels" / "val").is_dir(), "Pasta labels/val não encontrada. Ajuste DATASET_DIR."

## 2. Crie o arquivo `data.yaml`

O arquivo abaixo é gerado a partir das variáveis que você acabou de definir. Ele informa ao YOLO a localização das imagens e o nome de cada classe.

In [ ]:
import yaml

arquivo_config = Path("data.yaml")
configuracao = {
    "path": str(DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {indice: nome for indice, nome in enumerate(CLASS_NAMES)},
}

with arquivo_config.open("w", encoding="utf-8") as arquivo:
    yaml.safe_dump(configuracao, arquivo, allow_unicode=True, sort_keys=False)

print(arquivo_config.read_text(encoding="utf-8"))

Overwriting configs_modelo.yaml


## 3. Treine uma primeira versão

O YOLO usa `train` para ajustar o modelo e avaliar cada época com as imagens de validação. Comece com poucas épocas para confirmar que os caminhos, classes e labels estão corretos; depois aumente esse valor para um experimento completo.

In [ ]:
# Para a primeira execução, use poucas épocas e confirme se o pipeline funciona.
EPOCHS = 10
IMG_SIZE = 640
BATCH_SIZE = 16
NOME_EXPERIMENTO = "detector_custom_v1"

resultados = model.train(
    data=str(arquivo_config),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project="runs/detect",
    name=NOME_EXPERIMENTO,
)

PASTA_RESULTADOS = Path(resultados.save_dir)
print(f"Resultados salvos em: {PASTA_RESULTADOS}")

New https://pypi.org/project/ultralytics/8.3.153 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.134 🚀 Python-3.9.6 torch-2.3.0 CPU (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs_modelo.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=720, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_pothole2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ove

train: Scanning /Users/arnaldoalvesvianajunior/shift-fiap/10-lab18-yolo/dataset-pothole/dataset/train... 1562 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1562/1562 [00:00<00:00, 2661.57it/s]


train: New cache created: /Users/arnaldoalvesvianajunior/shift-fiap/10-lab18-yolo/dataset-pothole/dataset/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 121.4±22.3 MB/s, size: 31.1 KB)


val: Scanning /Users/arnaldoalvesvianajunior/shift-fiap/10-lab18-yolo/dataset-pothole/dataset/test... 421 images, 0 backgrounds, 0 corrupt: 100%|██████████| 421/421 [00:00<00:00, 2937.88it/s]

val: New cache created: /Users/arnaldoalvesvianajunior/shift-fiap/10-lab18-yolo/dataset-pothole/dataset/test.cache


Plotting labels to runs/detect/yolov8_pothole2/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)


2025/06/11 17:29:58 INFO mlflow.tracking.fluent: Experiment with name '/Shared/Ultralytics' does not exist. Creating a new experiment.
2025/06/11 17:29:59 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2025/06/11 17:29:59 WARNING mlflow.utils.autologging_utils: MLflow transformers autologging is known to be compatible with 4.35.2 <= transformers <= 4.51.2, but the installed version is 4.51.3. If you encounter errors during autologging, try upgrading / downgrading transformers to a compatible version, or try upgrading MLflow.
2025/06/11 17:30:02 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/06/11 17:30:09 INFO mlflow.tracking.fluent: Autologging successfully enabled for keras.
2025/06/11 17:30:09 INFO mlflow.tracking.fluent: Autologging successfully enabled for tensorflow.
2025/06/11 17:30:10 INFO mlflow.tracking.fluent: Autologging successfully enabled for transformers.


MLflow: logging run_id(9f8d1c2250d941ff880ff4c5d5fa221e) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 736 train, 736 val
Using 0 dataloader workers
Logging results to runs/detect/yolov8_pothole2
Starting training for 3 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


        1/3         0G      2.269      3.503      2.163         52        736:   5%|▌         | 5/98 [01:48<33:38, 21.70s/it]


KeyboardInterrupt: 

## 4. Inspecione as métricas e curvas

O treinamento salva gráficos para leitura do experimento. `results.png` reúne perdas e métricas por época; `F1_curve.png` ajuda a escolher um limiar de confiança equilibrado para o seu problema.

In [ ]:
import matplotlib.pyplot as plt

imagem_f1 = PASTA_RESULTADOS / "F1_curve.png"
plt.figure(figsize=(8, 5))
plt.imshow(plt.imread(imagem_f1))
plt.axis("off");
plt.show()

In [ ]:
imagem_resultados = PASTA_RESULTADOS / "results.png"
plt.figure(figsize=(12, 8))
plt.imshow(plt.imread(imagem_resultados))
plt.axis("off");
plt.show()

## 5. Teste o melhor modelo em uma imagem nova

Use uma imagem que não esteja em `images/train` nem em `images/val`. O arquivo `best.pt` é o checkpoint com melhor resultado de validação durante o treino.

In [ ]:
MODELO_TREINADO = PASTA_RESULTADOS / "weights" / "best.pt"

# Altere para uma imagem que não participou do treino nem da validação.
IMAGEM_TESTE = Path("/caminho/para/imagem_nova.jpg")

assert MODELO_TREINADO.is_file(), "best.pt não foi encontrado. Verifique o treinamento."
assert IMAGEM_TESTE.is_file(), "Imagem de teste não encontrada. Ajuste IMAGEM_TESTE."

In [ ]:
modelo_treinado = YOLO(MODELO_TREINADO)
resultados_teste = modelo_treinado.predict(
    source=str(IMAGEM_TESTE),
    conf=0.40,
    save=True,
)

In [ ]:
imagem_resultado = resultados_teste[0].plot()
plt.figure(figsize=(10, 7))
plt.imshow(imagem_resultado)
plt.axis("off");
plt.show()